# 07 — Retained Python plots and direct method comparisons

This notebook uses the source manifest written by notebook 06, so every figure is tied to the same verified five-dimension TEMPTED and MEFISTO runs.

It creates only the attached Python figures:

- `07_classification_metrics.png`
- `07_confusion_matrices.png`
- `07_latent_trajectory_comparison.png`
- `07_mefisto_factor_embedding.png`
- `07_paired_batch_performance.png`
- `07_runtime.png`
- `07_circular_greengenes_factor_weights.png`

and two new direct comparison figures:

- `07_subject_loading_direct_comparison.png`
- `07_feature_loading_direct_comparison.png`

MEFISTO signs are already aligned to matched TEMPTED components by notebook 06.


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio import Phylo
from matplotlib.colors import TwoSlopeNorm
from matplotlib.font_manager import FontProperties
from scipy.stats import mannwhitneyu

PRIMARY_METRIC = "balanced_accuracy"
CIRCULAR_TREE_GENERA = 16

LABEL_MAP = {
    "FIN": "FIN/EST",
    "FINLAND": "FIN/EST",
    "EST": "FIN/EST",
    "ESTONIA": "FIN/EST",
    "RUS": "RUS",
    "RUSSIA": "RUS",
}
GROUP_ORDER = ["FIN/EST", "RUS"]
GROUP_COLORS = {"FIN/EST": "#0072E8", "RUS": "#F03B20"}

def normalize_group(value):
    key = str(value).strip().upper()
    return LABEL_MAP.get(key, str(value).strip())

def display_dimension(method, column):
    number = str(column).split("_")[-1]
    return f"Component {number}" if method == "TEMPTED" else f"Factor {number}"

def factor_columns(frame):
    return sorted(
        [
            column for column in frame.columns
            if column.startswith("factor_")
            and column.split("_")[-1].isdigit()
            and pd.to_numeric(frame[column], errors="coerce").notna().any()
        ],
        key=lambda name: int(name.split("_")[-1]),
    )

def newest_plot_data(folder):
    runs = sorted(
        (path for path in folder.iterdir() if path.is_dir() and not path.name.startswith(".")),
        reverse=True,
    )
    required = [
        "source_manifest.csv",
        "factor_alignment.csv",
        "subject_scores.csv",
        "sample_factors.csv",
        "feature_loadings.csv",
        "trajectory_summary.csv",
        "loading_resamples.csv",
    ]
    for run in runs:
        if all((run / name).exists() for name in required):
            return run
    raise FileNotFoundError("Run notebook 06 first; no complete verified plot_data run was found.")

def existing_table(folder, stem):
    for path in [folder / f"{stem}.csv.gz", folder / f"{stem}.csv"]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {stem} in {folder}")

root = Path(".") if Path("data").exists() else Path("..")
plot_data_folder = newest_plot_data(root / "data" / "plot_data")
output = root / "data" / "figures" / plot_data_folder.name
output.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(plot_data_folder / "source_manifest.csv").set_index("item")["value"]
tempted_run = Path(manifest["tempted_run"])
mefisto_run = Path(manifest["mefisto_run"])
selected_batch = str(manifest["batch"])

subject_scores = pd.read_csv(plot_data_folder / "subject_scores.csv")
sample_factors = pd.read_csv(plot_data_folder / "sample_factors.csv")
feature_loadings = pd.read_csv(plot_data_folder / "feature_loadings.csv")
trajectory_summary = pd.read_csv(plot_data_folder / "trajectory_summary.csv")
loading_resamples = pd.read_csv(plot_data_folder / "loading_resamples.csv")
alignment = pd.read_csv(plot_data_folder / "factor_alignment.csv")

for frame in [subject_scores, sample_factors, trajectory_summary]:
    if "label" in frame.columns:
        frame["label"] = frame["label"].map(normalize_group)

# Hard validation: both methods must contribute five dimensions.
for table_name, frame in [
    ("subject scores", subject_scores),
    ("sample factors", sample_factors),
    ("feature loadings", feature_loadings),
]:
    for method in ["TEMPTED", "MEFISTO"]:
        dimensions = factor_columns(frame[frame["method"] == method])
        if len(dimensions) != 5:
            raise ValueError(
                f"{table_name}: expected 5 {method} dimensions, found {len(dimensions)}: {dimensions}"
            )

if len(alignment) != 5 or set(alignment["sign"]) - {-1, 1}:
    raise ValueError("factor_alignment.csv must contain exactly five finite sign assignments.")

# Performance tables are read directly from the same model runs. This avoids
# stale aggregate reports and guarantees 10 matched batches per method.
tempted_batches = {
    path.name for path in tempted_run.iterdir()
    if path.is_dir() and bool(re.fullmatch(r"batch_[0-9]+", path.name))
}
mefisto_batches = {
    path.name for path in mefisto_run.iterdir()
    if path.is_dir() and bool(re.fullmatch(r"batch_[0-9]+", path.name))
}
common_batches = sorted(tempted_batches & mefisto_batches)

if len(common_batches) != 10:
    raise ValueError(f"Expected 10 matched model batches; found {len(common_batches)}.")

metric_rows = []
prediction_rows = []

for method, run in [("TEMPTED", tempted_run), ("MEFISTO", mefisto_run)]:
    for batch in common_batches:
        folder = run / batch

        metric = pd.read_csv(folder / "metrics.csv").copy()
        metric["method"] = method
        metric["batch"] = batch
        metric_rows.append(metric)

        prediction = pd.read_csv(existing_table(folder, "predictions")).copy()
        prediction["method"] = method
        prediction["batch"] = batch
        prediction_rows.append(prediction)

metrics = pd.concat(metric_rows, ignore_index=True)
predictions = pd.concat(prediction_rows, ignore_index=True)

# Every batch must contain the same held-out subject count for both methods.
prediction_counts = predictions.groupby(["batch", "method"]).size().unstack()
if prediction_counts.isna().any().any() or (prediction_counts["TEMPTED"] != prediction_counts["MEFISTO"]).any():
    raise ValueError("TEMPTED and MEFISTO predictions are not paired on the same held-out subjects.")

predictions["truth"] = predictions["truth"].map(normalize_group)
predictions["predicted"] = predictions["predicted"].map(normalize_group)
confusion = (
    predictions.groupby(["method", "truth", "predicted"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

print("Plot data:", plot_data_folder)
print("TEMPTED:", tempted_run)
print("MEFISTO:", mefisto_run)
print("Matched batches:", len(common_batches))
print("Predictions per method:", predictions.groupby("method").size().to_dict())
print(alignment)


In [ ]:
def save_figure(figure, filename):
    figure.savefig(output / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(figure)

def robust_limits(values, padding=0.08, lower_quantile=0.01, upper_quantile=0.99):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1, 1)
    low, high = np.quantile(values, [lower_quantile, upper_quantile])
    if low == high:
        spread = max(abs(low), 1) * 0.1
        return low - spread, high + spread
    margin = (high - low) * padding
    return low - margin, high + margin

def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)
    valid = np.isfinite(p)
    if not valid.any():
        return adjusted
    valid_indices = np.where(valid)[0]
    order = valid_indices[np.argsort(p[valid])]
    running = 1.0
    for reverse_rank, index in enumerate(order[::-1], start=1):
        rank = len(order) - reverse_rank + 1
        running = min(running, p[index] * len(order) / rank)
        adjusted[index] = min(running, 1.0)
    return adjusted

def stars(p_value):
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""

def normalized_confusion(rows):
    matrix = rows.pivot(index="truth", columns="predicted", values="count").fillna(0)
    percentages = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0) * 100
    return matrix, percentages


## Held-out performance and classification


In [ ]:
metric_columns = [column for column in ["accuracy", "balanced_accuracy", "macro_f1"] if column in metrics]
figure, axes = plt.subplots(1, len(metric_columns), figsize=(4.6 * len(metric_columns), 4.6))
axes = np.atleast_1d(axes)

all_metric_values = metrics[metric_columns].to_numpy(float)
metric_low = max(0, np.nanmin(all_metric_values) - 0.08)
metric_high = min(1, np.nanmax(all_metric_values) + 0.08)

for axis, metric_name in zip(axes, metric_columns):
    methods = [method for method in ["TEMPTED", "MEFISTO"] if method in metrics["method"].unique()]
    values = [metrics.loc[metrics["method"] == method, metric_name].dropna() for method in methods]
    axis.boxplot(values, tick_labels=methods, widths=0.55)
    axis.set_title(metric_name.replace("_", " ").title())
    axis.set_ylim(metric_low, metric_high)
    axis.set_ylabel("Held-out score")
    axis.grid(axis="y", alpha=0.25)

figure.suptitle("Held-out subject classification across repeated splits")
figure.tight_layout()
save_figure(figure, "07_classification_metrics.png")

paired = metrics.pivot_table(index="batch", columns="method", values=PRIMARY_METRIC).dropna()
if {"TEMPTED", "MEFISTO"}.issubset(paired.columns):
    figure, axis = plt.subplots(figsize=(6.2, 5))
    for _, row in paired.iterrows():
        axis.plot(["TEMPTED", "MEFISTO"], [row["TEMPTED"], row["MEFISTO"]], marker="o", alpha=0.5)
    axis.set_ylim(metric_low, metric_high)
    axis.set_ylabel(PRIMARY_METRIC.replace("_", " ").title())
    axis.set_title("Paired held-out performance by split")
    axis.grid(axis="y", alpha=0.25)
    save_figure(figure, "07_paired_batch_performance.png")

methods = [method for method in ["TEMPTED", "MEFISTO"] if method in confusion["method"].unique()]
figure, axes = plt.subplots(1, len(methods), figsize=(5.2 * len(methods), 4.6))
axes = np.atleast_1d(axes)

for axis, method in zip(axes, methods):
    counts, percentages = normalized_confusion(confusion[confusion["method"] == method])
    image = axis.imshow(percentages, vmin=0, vmax=100)
    axis.set_xticks(range(len(percentages.columns)), percentages.columns)
    axis.set_yticks(range(len(percentages.index)), percentages.index)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Observed")
    axis.set_title(method)
    for y in range(percentages.shape[0]):
        for x in range(percentages.shape[1]):
            pct = percentages.iloc[y, x]
            count = counts.iloc[y, x]
            axis.text(x, y, f"{pct:.0f}%\n(n={int(count)})", ha="center", va="center", fontsize=8)

figure.colorbar(image, ax=list(axes), fraction=0.025, pad=0.04, label="Row percentage")
figure.suptitle("Aggregated held-out confusion matrices")
figure.subplots_adjust(top=0.84, wspace=0.35)
save_figure(figure, "07_confusion_matrices.png")

figure, axis = plt.subplots(figsize=(6.2, 4.8))
runtime = [metrics.loc[metrics["method"] == method, "elapsed_seconds"].dropna() for method in methods]
axis.boxplot(runtime, tick_labels=methods)
axis.set_yscale("log")
axis.set_ylabel("Seconds per split (log scale)")
axis.set_title("Runtime per train/test split")
axis.grid(axis="y", alpha=0.25)
save_figure(figure, "07_runtime.png")


## Latent representations


In [ ]:
factor_columns = [column for column in sample_factors.columns if column.startswith("factor_")]

# MEFISTO: fitted factor values, as in the microbiome application in Fig. 3a-b.
mefisto_samples = sample_factors[sample_factors["method"] == "MEFISTO"].copy()
if len(factor_columns) >= 2 and len(mefisto_samples):
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for label in GROUP_ORDER:
        rows = mefisto_samples[mefisto_samples["label"] == label]
        if len(rows):
            axes[0].scatter(
                rows[factor_columns[0]], rows[factor_columns[1]],
                s=14, alpha=0.42, label=label,
                color=GROUP_COLORS[label], rasterized=True,
            )
    axes[0].set_xlabel("Factor 1")
    axes[0].set_ylabel("Factor 2")
    axes[0].set_title("MEFISTO factors by phenotype")
    axes[0].legend(markerscale=2)

    scatter = axes[1].scatter(
        mefisto_samples[factor_columns[0]],
        mefisto_samples[factor_columns[1]],
        c=mefisto_samples["time"],
        s=11,
        alpha=0.35,
        rasterized=True,
    )
    axes[1].set_xlabel("Factor 1")
    axes[1].set_ylabel("Factor 2")
    axes[1].set_title("MEFISTO factors by age")
    figure.colorbar(scatter, ax=axes[1], label="Age at collection")

    for axis in axes:
        axis.set_xlim(*robust_limits(mefisto_samples[factor_columns[0]]))
        axis.set_ylim(*robust_limits(mefisto_samples[factor_columns[1]]))
        axis.grid(alpha=0.15)

    figure.tight_layout()
    save_figure(figure, "07_mefisto_factor_embedding.png")

# Longitudinal curves use the sign-aligned values from notebook 06. Free y-scales
# preserve the different model-specific scales.
methods = [method for method in ["TEMPTED", "MEFISTO"] if method in trajectory_summary["method"].unique()]
factor_sets = [
    set(trajectory_summary.loc[trajectory_summary["method"] == method, "factor"])
    for method in methods
]
factors = sorted(set.intersection(*factor_sets), key=lambda x: int(str(x).split("_")[-1]))
if len(factors) != 5:
    raise ValueError(f"Trajectory comparison requires 5 dimensions in both methods; found {factors}")
figure, axes = plt.subplots(len(methods), len(factors), figsize=(5.2 * len(factors), 4.2 * len(methods)), squeeze=False)

for row_index, method in enumerate(methods):
    for column_index, factor in enumerate(factors):
        axis = axes[row_index, column_index]
        rows = trajectory_summary[
            (trajectory_summary["method"] == method) &
            (trajectory_summary["factor"] == factor)
        ]
        for label in GROUP_ORDER:
            group = rows[rows["label"] == label].sort_values("time")
            if len(group):
                axis.plot(
                    group["time"], group["median"], label=label,
                    color=GROUP_COLORS[label], linewidth=2.0,
                )
                axis.fill_between(
                    group["time"], group["lower"], group["upper"],
                    color=GROUP_COLORS[label], alpha=0.22,
                )
        axis.set_title(f"{method} — {display_dimension(method, factor)}")
        axis.set_xlabel("Age at collection")
        axis.set_ylabel(
            "TEMPTED sample trajectory" if method == "TEMPTED"
            else "MEFISTO factor value"
        )
        axis.grid(alpha=0.2)
        if len(rows):
            axis.set_ylim(*robust_limits(np.r_[rows["lower"], rows["upper"]], padding=0.12, lower_quantile=0, upper_quantile=1))
        axis.legend()

figure.suptitle("Longitudinal latent representations")
figure.tight_layout()
save_figure(figure, "07_latent_trajectory_comparison.png")


## Direct TEMPTED–MEFISTO loading comparisons


In [ ]:
# Direct cross-method comparisons use the loading-based component/factor matching
# from notebook 06. Values are standardized within each method because TEMPTED
# and MEFISTO use different latent scales.

def zscore(values):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    mean = np.nanmean(values)
    sd = np.nanstd(values)
    if not np.isfinite(sd) or sd == 0:
        return values - mean
    return (values - mean) / sd

def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    if keep.sum() < 3 or np.std(x[keep]) == 0 or np.std(y[keep]) == 0:
        return np.nan
    return np.corrcoef(x[keep], y[keep])[0, 1]

# Subject-level comparison: TEMPTED subject score versus the aligned MEFISTO
# mean factor value for the same held-out subject.
figure, axes = plt.subplots(1, 5, figsize=(22, 4.4))
for axis, row in zip(axes, alignment.itertuples(index=False)):
    tempted_factor = row.tempted_component
    mefisto_factor = row.mefisto_factor

    left = (
        subject_scores[subject_scores["method"] == "TEMPTED"]
        [["subject_id", "label", tempted_factor]]
        .rename(columns={tempted_factor: "tempted"})
    )
    right = (
        subject_scores[subject_scores["method"] == "MEFISTO"]
        [["subject_id", mefisto_factor]]
        .rename(columns={mefisto_factor: "mefisto"})
    )
    joined = left.merge(right, on="subject_id", how="inner")
    joined["tempted_z"] = zscore(joined["tempted"])
    joined["mefisto_z"] = zscore(joined["mefisto"])

    for label in GROUP_ORDER:
        rows = joined[joined["label"] == label]
        axis.scatter(
            rows["tempted_z"], rows["mefisto_z"],
            s=22, alpha=0.65, label=label,
            color=GROUP_COLORS[label],
        )

    r = finite_corr(joined["tempted_z"], joined["mefisto_z"])
    low = min(*robust_limits(np.r_[joined["tempted_z"], joined["mefisto_z"]]))
    high = max(*robust_limits(np.r_[joined["tempted_z"], joined["mefisto_z"]]))
    axis.plot([low, high], [low, high], "--", linewidth=1)
    axis.set_xlim(low, high)
    axis.set_ylim(low, high)
    axis.set_title(
        f"{display_dimension('TEMPTED', tempted_factor)} vs "
        f"{display_dimension('MEFISTO', mefisto_factor)}\nr={r:.2f}"
    )
    axis.set_xlabel("TEMPTED standardized subject score")
    axis.set_ylabel("MEFISTO standardized mean factor")
    axis.grid(alpha=0.2)

axes[0].legend()
figure.suptitle("Direct subject-level loading comparison")
figure.tight_layout()
save_figure(figure, "07_subject_loading_direct_comparison.png")

# Feature-level comparison: the same genera/features under the matched and
# sign-aligned TEMPTED component and MEFISTO factor.
figure, axes = plt.subplots(1, 5, figsize=(22, 4.4))
for axis, row in zip(axes, alignment.itertuples(index=False)):
    tempted_factor = row.tempted_component
    mefisto_factor = row.mefisto_factor

    left = (
        feature_loadings[feature_loadings["method"] == "TEMPTED"]
        [["feature_id", tempted_factor]]
        .rename(columns={tempted_factor: "tempted"})
    )
    right = (
        feature_loadings[feature_loadings["method"] == "MEFISTO"]
        [["feature_id", mefisto_factor]]
        .rename(columns={mefisto_factor: "mefisto"})
    )
    joined = left.merge(right, on="feature_id", how="inner")
    joined["tempted_z"] = zscore(joined["tempted"])
    joined["mefisto_z"] = zscore(joined["mefisto"])

    r = finite_corr(joined["tempted_z"], joined["mefisto_z"])
    axis.scatter(joined["tempted_z"], joined["mefisto_z"], s=22, alpha=0.65)
    low = min(*robust_limits(np.r_[joined["tempted_z"], joined["mefisto_z"]]))
    high = max(*robust_limits(np.r_[joined["tempted_z"], joined["mefisto_z"]]))
    axis.plot([low, high], [low, high], "--", linewidth=1)
    axis.set_xlim(low, high)
    axis.set_ylim(low, high)
    axis.set_title(
        f"{display_dimension('TEMPTED', tempted_factor)} vs "
        f"{display_dimension('MEFISTO', mefisto_factor)}\nr={r:.2f}"
    )
    axis.set_xlabel("TEMPTED standardized feature loading")
    axis.set_ylabel("MEFISTO standardized feature loading")
    axis.grid(alpha=0.2)

figure.suptitle("Direct feature-loading comparison")
figure.tight_layout()
save_figure(figure, "07_feature_loading_direct_comparison.png")


## Circular Greengenes phylogenetic factor-weight plot


In [ ]:
from copy import deepcopy

GREEN_GENES_TREE = root / "data" / "DIABIMMUNE" / "97_otus.tree"
GREEN_GENES_TAXONOMY = root / "data" / "DIABIMMUNE" / "97_otu_taxonomy.txt"

def canonical_taxonomy(text):
    return "|".join(part.strip() for part in str(text).replace(";", "|").split("|") if part.strip())

def genus_path(text):
    parts = canonical_taxonomy(text).split("|")
    genus_index = next((index for index, part in enumerate(parts) if part.startswith("g__")), None)
    if genus_index is None:
        return None
    return "|".join(parts[: genus_index + 1])


taxonomy = pd.read_csv(GREEN_GENES_TAXONOMY, sep="\t", names=["otu_id", "taxonomy"], dtype=str)
taxonomy["genus_path"] = taxonomy["taxonomy"].map(genus_path)

# Use the verified, sign-aligned model loadings written by notebook 06.
tree_feature_loadings = feature_loadings.copy()
tree_loading_resamples = loading_resamples.copy()

print("Phylogenetic tree source runs:")
print("  TEMPTED:", tempted_run)
print("  MEFISTO:", mefisto_run)
print("  TEMPTED components:", factor_columns(tree_feature_loadings[tree_feature_loadings["method"] == "TEMPTED"]))
print("  MEFISTO factors:", factor_columns(tree_feature_loadings[tree_feature_loadings["method"] == "MEFISTO"]))

# Include every fitted dimension. TEMPTED calls these components; MEFISTO
# calls them factors. The shared factor_N column names are only an interchange
# convention used by the workflow.
def dimensions_for_method(method):
    rows = tree_feature_loadings[tree_feature_loadings["method"] == method]
    return [
        column for column in rows.columns
        if column.startswith("factor_")
        and pd.to_numeric(rows[column], errors="coerce").notna().any()
    ]

dimensions_by_method = {
    method: dimensions_for_method(method)
    for method in ["TEMPTED", "MEFISTO"]
    if method in tree_feature_loadings["method"].unique()
}

tree_factors = sorted(
    set().union(*dimensions_by_method.values()),
    key=lambda name: int(name.split("_")[-1]),
)
genus_loadings = tree_feature_loadings[
    tree_feature_loadings["feature_id"].str.contains(r"\|g__", regex=True, na=False)
    & ~tree_feature_loadings["feature_id"].str.contains(r"\|s__", regex=True, na=False)
].copy()
genus_loadings["genus_path"] = genus_loadings["feature_id"].map(genus_path)
genus_loadings["strength"] = genus_loadings[tree_factors].apply(pd.to_numeric, errors="coerce").abs().max(axis=1)

selected = (
    genus_loadings.sort_values(["method", "strength"], ascending=[True, False])
    .groupby("method", group_keys=False)
    .head(CIRCULAR_TREE_GENERA)
)
selected_paths = selected["genus_path"].dropna().drop_duplicates()

representatives = (
    taxonomy[taxonomy["genus_path"].isin(selected_paths)]
    .dropna(subset=["genus_path"])
    .drop_duplicates("genus_path")[["genus_path", "otu_id"]]
)
selected = selected.merge(representatives, on="genus_path", how="inner")

full_tree = Phylo.read(GREEN_GENES_TREE, "newick")
tree_tip_names = {tip.name for tip in full_tree.get_terminals()}
selected = selected[selected["otu_id"].isin(tree_tip_names)].copy()

# Resampling-based genus enrichment.
enrichment_rows = []
if len(tree_loading_resamples):
    resamples = tree_loading_resamples[
        tree_loading_resamples["feature_id"].str.contains(r"\|g__", regex=True, na=False)
        & ~tree_loading_resamples["feature_id"].str.contains(r"\|s__", regex=True, na=False)
    ].copy()
    resamples["genus_path"] = resamples["feature_id"].map(genus_path)

    for method in resamples["method"].drop_duplicates():
        method_rows = resamples[resamples["method"] == method]
        for factor in dimensions_by_method.get(method, []):
            all_values = pd.to_numeric(method_rows[factor], errors="coerce")
            for genus, genus_rows in method_rows.groupby("genus_path"):
                values = pd.to_numeric(genus_rows[factor], errors="coerce").dropna().to_numpy()
                if len(values) < 3:
                    continue
                median_value = float(np.median(values))
                if median_value > 0:
                    genus_values = values[values > 0]
                    background = all_values[(all_values > 0) & (method_rows["genus_path"] != genus)].dropna().to_numpy()
                    alternative = "greater"
                    direction = "positive"
                elif median_value < 0:
                    genus_values = values[values < 0]
                    background = all_values[(all_values < 0) & (method_rows["genus_path"] != genus)].dropna().to_numpy()
                    alternative = "less"
                    direction = "negative"
                else:
                    continue

                if len(genus_values) < 3 or len(background) < 3:
                    continue

                p_value = mannwhitneyu(genus_values, background, alternative=alternative).pvalue
                enrichment_rows.append({
                    "method": method,
                    "factor": factor,
                    "genus_path": genus,
                    "direction": direction,
                    "n_batches": len(values),
                    "median_loading": median_value,
                    "raw_p": p_value,
                })

enrichment = pd.DataFrame(enrichment_rows)
if len(enrichment):
    enrichment["adjusted_p"] = np.nan
    for _, indices in enrichment.groupby(["method", "factor", "direction"]).groups.items():
        enrichment.loc[indices, "adjusted_p"] = bh_adjust(enrichment.loc[indices, "raw_p"].to_numpy())
    enrichment["stars"] = enrichment["adjusted_p"].map(stars)
else:
    enrichment = pd.DataFrame(columns=["method", "factor", "genus_path", "direction", "n_batches", "median_loading", "raw_p", "adjusted_p", "stars"])

enrichment.to_csv(plot_data_folder / "07_circular_greengenes_enrichment_tests.csv", index=False)

representative_tip_by_path = (
    selected[["genus_path", "otu_id"]].drop_duplicates("genus_path")
    .set_index("genus_path")["otu_id"].to_dict()
)
path_by_representative_tip = {otu_id: path for path, otu_id in representative_tip_by_path.items()}

pruned_tree = deepcopy(full_tree)
tips_to_keep = set(path_by_representative_tip)
for tip in list(pruned_tree.get_terminals()):
    if tip.name not in tips_to_keep:
        pruned_tree.prune(tip)

tips = pruned_tree.get_terminals()
tip_angles = {tip: 2 * np.pi * index / len(tips) for index, tip in enumerate(tips)}

def circular_mean(angles):
    return np.angle(np.mean(np.exp(1j * np.asarray(angles)))) % (2 * np.pi)

node_angles = {}
def assign_node_angle(clade):
    if clade in tip_angles:
        angle = tip_angles[clade]
    else:
        angle = circular_mean([assign_node_angle(child) for child in clade.clades])
    node_angles[clade] = angle
    return angle

assign_node_angle(pruned_tree.root)
depths = pruned_tree.depths(unit_branch_lengths=True)
maximum_depth = max(depths.values())
node_radii = {clade: 0.12 + 0.53 * depth / maximum_depth for clade, depth in depths.items()}

all_values = selected[tree_factors].to_numpy(float)
scale_limit = np.nanmax(np.abs(all_values))
if not np.isfinite(scale_limit) or scale_limit == 0:
    scale_limit = 1.0
color_norm = TwoSlopeNorm(vmin=-scale_limit, vcenter=0, vmax=scale_limit)
color_map = plt.get_cmap("RdBu_r")

def draw_tree_branches(axis):
    for parent in pruned_tree.find_clades(order="level"):
        if not parent.clades:
            continue
        parent_angle = node_angles[parent]
        parent_radius = node_radii[parent]
        child_angles = np.asarray([node_angles[child] for child in parent.clades])
        unwrapped = np.unwrap(np.concatenate(([parent_angle], child_angles)))[1:]
        arc_angles = np.linspace(unwrapped.min(), unwrapped.max(), 40)
        axis.plot(arc_angles, np.full_like(arc_angles, parent_radius), color="black", linewidth=0.65)
        for child, child_angle in zip(parent.clades, unwrapped):
            axis.plot([child_angle, child_angle], [parent_radius, node_radii[child]], color="black", linewidth=0.65)

def draw_text_on_circle(axis, center_angle, radius, text, fontsize=6.6, tracking=0.6, angular_slot=None):
    text = str(text).strip()
    if not text:
        return
    figure = axis.figure
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    axis_radius_pixels = min(axis.bbox.width, axis.bbox.height) / 2
    circle_radius_pixels = max(axis_radius_pixels * radius / axis.get_ylim()[1], 1)
    font = FontProperties(size=fontsize)
    measured_width = renderer.get_text_width_height_descent(text, font, ismath=False)[0]
    if angular_slot is not None:
        available_width = circle_radius_pixels * angular_slot * 0.82
        if measured_width > available_width:
            fontsize = max(3.8, fontsize * available_width / measured_width)
            font = FontProperties(size=fontsize)

    advances = []
    for character in text:
        width = renderer.get_text_width_height_descent("n" if character == " " else character, font, ismath=False)[0]
        if character == " ":
            width *= 0.45
        advances.append(width + tracking * fontsize / 6.6)

    total = sum(advances)
    centers = np.cumsum(advances) - np.asarray(advances) / 2
    offsets = (centers - total / 2) / circle_radius_pixels
    normalized = center_angle % (2 * np.pi)
    reverse = np.pi / 2 < normalized < 3 * np.pi / 2
    direction = -1 if reverse else 1
    characters = text[::-1] if reverse else text
    character_offsets = offsets[::-1] if reverse else offsets

    for character, offset in zip(characters, character_offsets):
        angle = center_angle + direction * offset
        rotation = -np.degrees(angle) + (180 if reverse else 0)
        axis.text(angle, radius, character, rotation=rotation, rotation_mode="anchor",
                  ha="center", va="center", fontsize=fontsize, clip_on=False)

# Ring geometry expands with the number of available dimensions.
ring_inner = 0.94
ring_width = 0.095
ring_gap = 0.018

def draw_method_tree(axis, method):
    method_rows = selected[selected["method"] == method].drop_duplicates("genus_path").set_index("genus_path")
    method_factors = dimensions_by_method.get(method, [])
    method_enrichment = enrichment[enrichment["method"] == method] if len(enrichment) else enrichment

    axis.set_theta_direction(-1)
    axis.set_theta_offset(np.pi / 2)
    axis.set_axis_off()
    outer_radius = ring_inner + len(method_factors) * (ring_width + ring_gap)
    axis.set_ylim(0, outer_radius + 0.12)
    draw_tree_branches(axis)

    label_radius = 0.80
    angular_width = 2 * np.pi / len(tips) * 0.96

    for tip in tips:
        angle = node_angles[tip]
        path = path_by_representative_tip[tip.name]
        genus = path.split("|")[-1].replace("g__", "") or "Unclassified"
        axis.plot([angle, angle], [node_radii[tip], label_radius - 0.015], color="black", linewidth=0.45)
        draw_text_on_circle(axis, angle, label_radius, genus, fontsize=6.8, tracking=0.38, angular_slot=2 * np.pi / len(tips))

    for factor_number, factor in enumerate(method_factors):
        bottom = ring_inner + factor_number * (ring_width + ring_gap)
        for tip in tips:
            angle = node_angles[tip]
            path = path_by_representative_tip[tip.name]
            value = float(method_rows.loc[path, factor]) if path in method_rows.index else np.nan
            face_color = color_map(color_norm(value)) if np.isfinite(value) else (0.93, 0.93, 0.93, 1)
            axis.bar(angle, ring_width, width=angular_width, bottom=bottom,
                     color=face_color, edgecolor="white", linewidth=0.45, align="center")

            match = method_enrichment[
                (method_enrichment["factor"] == factor) &
                (method_enrichment["genus_path"] == path)
            ]
            star_text = match["stars"].iloc[0] if len(match) else ""
            if star_text:
                draw_text_on_circle(
                    axis,
                    angle,
                    bottom + ring_width / 2,
                    star_text,
                    fontsize=9.8,
                    tracking=0.16,
                    angular_slot=angular_width * 0.82,
                )

        axis.text(np.deg2rad(333), bottom + ring_width / 2,
                  display_dimension(method, factor), ha="left", va="center", fontsize=9)

    axis.text(0, 0, method, ha="center", va="center", fontsize=12, fontweight="bold")

print("Circular-tree dimensions:")
for method, dimensions in dimensions_by_method.items():
    print(f"  {method}: {len(dimensions)} -> {dimensions}")

methods_for_tree = [method for method in ["TEMPTED", "MEFISTO"] if method in selected["method"].unique()]
maximum_rings = max((len(v) for v in dimensions_by_method.values()), default=1)
tree_figure_height = 10.5 + max(0, maximum_rings - 2) * 0.7
figure, axes = plt.subplots(
    1, len(methods_for_tree),
    figsize=(tree_figure_height * len(methods_for_tree), tree_figure_height),
                           subplot_kw={"projection": "polar"})
axes = np.atleast_1d(axes)
for axis, method in zip(axes, methods_for_tree):
    draw_method_tree(axis, method)

color_scalar = plt.cm.ScalarMappable(norm=color_norm, cmap=color_map)
legend_axis = figure.add_axes([0.92, 0.23, 0.018, 0.52])
colorbar = figure.colorbar(color_scalar, cax=legend_axis)
colorbar.set_label("TEMPTED component / MEFISTO factor loading")
figure.suptitle("Greengenes circular phylogenetic component/factor-weight plots", fontsize=15)
figure.subplots_adjust(top=0.90, wspace=0.28, right=0.88)
figure.savefig(output / "07_circular_greengenes_factor_weights.png", dpi=220)
plt.show()
plt.close(figure)

selected[["method", "feature_id", "genus_path", "otu_id", *tree_factors]].to_csv(
    plot_data_folder / "07_circular_greengenes_feature_mapping.csv", index=False
)

print("Significant circular-tree cells:", int((enrichment["stars"] != "").sum()) if len(enrichment) else 0)
if not len(tree_loading_resamples):
    print("No direct model loading resamples were available for enrichment stars.")


## Output audit


In [ ]:
expected_files = [
    "07_classification_metrics.png",
    "07_confusion_matrices.png",
    "07_latent_trajectory_comparison.png",
    "07_mefisto_factor_embedding.png",
    "07_paired_batch_performance.png",
    "07_runtime.png",
    "07_circular_greengenes_factor_weights.png",
    "07_subject_loading_direct_comparison.png",
    "07_feature_loading_direct_comparison.png",
]

audit = pd.DataFrame([
    {
        "file": filename,
        "created": (output / filename).exists(),
        "bytes": (output / filename).stat().st_size if (output / filename).exists() else 0,
    }
    for filename in expected_files
])
audit.to_csv(plot_data_folder / "plot_audit.csv", index=False)
display(audit)
print("Finished:", output)
